# Client Folder Structure Creator

Run this from `runscripts/`; don't copy it into a client's folder. It lays out the client's folder, which every later runscript requires, and copies in the runscripts with cells to edit for that client. Steps 1 and 2 have nothing to edit, so they run from the repo.

Client folders go in `python/lab/client_work/`, which git ignores. Set `CLIENT_DATA_FOLDER` in the repo's `.env` to put them somewhere else.

Configure the variables in the first code cell and run the notebook.

Options:
- period: 'baseline' | 'pilot'
- data_type: 'procurement' | 'serving' | 'both'
- data_format: 'pdf' | 'tabular' | 'both'

A pilot starts from the baseline's copies of the runscripts, edits included, where the baseline has them. A copy the client already has is never overwritten.

# Setup

In [ ]:
import os
import shutil
from pathlib import Path
from typing import Literal

from gbd_foodservice_insights_lab import PACKAGE_DIR
from gbd_foodservice_insights_lab.notebook_runscript_setup import (
    COMPARE_PROCUREMENT_SERVING_RUNSCRIPT,
    get_customer_template_dir,
    load_repo_env,
    runscripts_for_data_folder,
)

load_repo_env()

In [ ]:
# === CLIENT INFORMATION ===
client_name = ""  # <-- set to actual client name (string)
sub_client = None  # e.g ["Vendor_1", "Vendor_2"] or None

# === PERIOD AND DATA SETTINGS ===
period: Literal["baseline", "pilot"] = "baseline"  # 'baseline' | 'pilot'
data_type = "serving"  # 'procurement' | 'serving' | 'both'
data_format: Literal["pdf", "tabular", "both"] = "tabular"  # 'pdf' | 'tabular' | 'both'

# === OPTIONAL FEATURES ===
include_initial_data_check = False  # Include exploratory notebook (0 initial_data_check.ipynb)
dry_run = False  # Set to True to preview without creating files/folders

# === PATHS ===
client_data_folder = os.getenv("CLIENT_DATA_FOLDER") or PACKAGE_DIR.parent / "client_work"

In [ ]:
# Validation sets
VALID_PERIODS = {"baseline", "pilot"}
VALID_DATA_TYPES = {"procurement", "purchasing", "POS", "point of sale", "sales", "serving", "both"}
VALID_DATA_FORMATS = {"pdf", "tabular", "both"}

DATA_DIR_MAP = {
    "procurement": ["purchasing-procurement data"],
    "purchasing": ["purchasing-procurement data"],
    "serving": ["sales-serving data"],
    "POS": ["sales-serving data"],
    "point of sale": ["sales-serving data"],
    "sales": ["sales-serving data"],
    "both": ["purchasing-procurement data", "sales-serving data"],
}

# Validate configuration


def validate_config():
    """Validate all configuration parameters."""
    errors = []

    if not client_name.strip():
        errors.append("client_name cannot be empty")
    if period not in VALID_PERIODS:
        errors.append(f"period must be one of {sorted(VALID_PERIODS)}")
    if data_type not in VALID_DATA_TYPES:
        errors.append(f"data_type must be one of {sorted(VALID_DATA_TYPES)}")
    if data_format not in VALID_DATA_FORMATS:
        errors.append(f"data_format must be one of {sorted(VALID_DATA_FORMATS)}")
    if sub_client is not None and not (
        isinstance(sub_client, list)
        and all(isinstance(sc, str) and sc.strip() for sc in sub_client)
    ):
        errors.append("sub_client must be None or a list of non-empty strings")

    if errors:
        raise ValueError("❌ Configuration errors:\n  • " + "\n  • ".join(errors))


validate_config()

# Expand configuration
client_data_folder_expanded = Path(client_data_folder).expanduser()
template_dir = get_customer_template_dir()
client_dir = client_data_folder_expanded / client_name
data_formats = ["pdf", "tabular"] if data_format == "both" else [data_format]
sub_clients = sub_client if sub_client is not None else [None]

print("✅ Configuration validated successfully!")
# Track created vs existing items
created_dirs = []
existing_dirs = []
created_notebooks = []
existing_notebooks = []

if dry_run:
    print("🔍 DRY RUN MODE - No files or folders will be created\n")

print(f"Client folder will be created at: {client_dir}")
print(f"Copying runscripts from: {template_dir}")
if sub_client is not None:
    print(f"Sub-clients: {sub_client}")
print()

# Functions

In [ ]:
def ensure_dir(path):
    """Create directory if it doesn't exist, track the result."""
    if path.exists():
        existing_dirs.append(path)
    else:
        if not dry_run:
            path.mkdir(parents=True)
        created_dirs.append(path)


def copy_runscript(source, dest):
    """Copy a runscript unless the client already has a copy, which may hold edits."""
    if dest.exists():
        existing_notebooks.append(dest)
        return
    if not dry_run:
        shutil.copy2(source, dest)
    created_notebooks.append(dest)

# Make the folders

In [ ]:
# Create directory structure
ensure_dir(client_dir)

period_root = client_dir / period
ensure_dir(period_root)

for data_folder in DATA_DIR_MAP[data_type]:
    data_root = period_root / data_folder
    ensure_dir(data_root)

    for sub_client_name in sub_clients:
        # If sub_client is specified, create sub-client folder level
        if sub_client_name is not None:
            current_data_root = data_root / sub_client_name
            ensure_dir(current_data_root)
        else:
            current_data_root = data_root

        raw_data_dir = current_data_root / "raw_data"
        ensure_dir(raw_data_dir)

        # A pilot starts from the baseline's copies, which hold its edits, where they exist.
        baseline_data_root = client_dir / "baseline" / current_data_root.relative_to(period_root)
        for name in runscripts_for_data_folder(
            period, data_formats, include_initial_check=include_initial_data_check
        ):
            baseline_copy = baseline_data_root / name
            use_baseline = period == "pilot" and baseline_copy.exists()
            copy_runscript(
                baseline_copy if use_baseline else template_dir / name, current_data_root / name
            )

# Step 3 compares the baseline's procurement with its serving data, so it sits above both.
if period == "baseline" and len(DATA_DIR_MAP[data_type]) == 2:
    copy_runscript(
        template_dir / COMPARE_PROCUREMENT_SERVING_RUNSCRIPT,
        period_root / COMPARE_PROCUREMENT_SERVING_RUNSCRIPT,
    )

# Display results
print("=" * 50)
print("DIRECTORY CREATION RESULTS")
print("=" * 50)
for path in created_dirs:
    print(f"✓ Created: {path}")
for path in existing_dirs:
    print(f"• Already exists: {path}")

if created_notebooks or existing_notebooks:
    print("\n" + "=" * 50)
    print("NOTEBOOK COPY RESULTS")
    print("=" * 50)
    for path in created_notebooks:
        print(f"✓ Copied: {path}")
    for path in existing_notebooks:
        print(f"• Already exists: {path}")

# Summary statistics
print("\n" + "=" * 50)
print("📊 SUMMARY")
print("=" * 50)
print(f"   Directories created: {len(created_dirs)}")
print(f"   Directories already existed: {len(existing_dirs)}")
print(f"   Notebooks copied: {len(created_notebooks)}")
print(f"   Notebooks already existed: {len(existing_notebooks)}")

if dry_run:
    print("\n🔍 DRY RUN - No actual changes were made")
else:
    print("\n✅ Setup complete!")

In [ ]:
def print_directory_tree(root: Path, max_depth: int = 3, prefix: str = ""):
    """Print a simple directory tree structure including files."""
    if max_depth < 0:
        return

    entries = sorted(root.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))
    for i, entry in enumerate(entries):
        is_last = i == len(entries) - 1
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{entry.name}")
        if entry.is_dir():
            next_prefix = prefix + ("    " if is_last else "│   ")
            print_directory_tree(entry, max_depth - 1, next_prefix)


print(f"\n=== Directory Structure for {client_name} ===")
print(client_dir.name)
print_directory_tree(client_dir, max_depth=3)